# Phase 4: Ensemble Detection Layer Analysis
## UNSW-NB15 ML-SIEM

Weighted voting ensemble aggregating Random Forest, XGBoost, Isolation Forest, and Dense Autoencoder.

| Model | Type | Weight |
|---|---|---|
| Random Forest | Supervised | 0.35 |
| XGBoost | Supervised | 0.35 |
| Isolation Forest | Unsupervised | 0.15 |
| Dense Autoencoder | Unsupervised | 0.15 |

**Formula:** `score = 0.35×RF + 0.35×XGB + 0.15×IF + 0.15×AE` → ATTACK if score ≥ 0.50


In [ ]:
import sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.ensemble import EnsembleDetector, ATTACK_THRESHOLD

plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False})
print("Imports OK — PROJECT_ROOT:", PROJECT_ROOT)


## 1. Load Evaluation Report

In [ ]:
EVAL_PATH = PROJECT_ROOT / 'reports' / 'ensemble_evaluation.json'
assert EVAL_PATH.exists(), f"Run run_ensemble.py first: {EVAL_PATH}"

with open(EVAL_PATH) as f:
    report = json.load(f)

metrics = report['ensemble']
meta    = report['metadata']
indiv   = metrics.get('individual_model_comparison', {})

print(f"Test samples: {meta['test_rows']:,} | Attack: {meta['n_attack']:,} | Normal: {meta['n_normal']:,}")
print()
for k in ['accuracy','precision','recall','f1','false_positive_rate']:
    print(f"  {k:26s}: {metrics[k]:.4f}")


## 2. Load Sample for Detailed Vote Analysis

In [ ]:
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
feat_names = (PROCESSED/'feature_list.txt').read_text().strip().splitlines()
df_test    = pd.read_parquet(PROCESSED/'test.parquet', columns=feat_names+['label','attack_cat'])
df_s       = df_test.sample(n=20_000, random_state=42).reset_index(drop=True)
X_s        = df_s[feat_names].values.astype('float32')
y_s        = (df_s['label']==1).astype(int).values
print(f"Sample: {len(df_s):,} | attack={y_s.sum():,} | normal={(y_s==0).sum():,}")


In [ ]:
ensemble = EnsembleDetector.from_disk(
    PROJECT_ROOT/'data/models/rf_detector.joblib',
    PROJECT_ROOT/'data/models/xgb_detector.joblib',
    PROJECT_ROOT/'models/isolation_forest.joblib',
    PROJECT_ROOT/'models/autoencoder.pt',
    batch_size=2048
)
results = ensemble.predict(X_s)

rows = []
for i, r in enumerate(results):
    vm = {v.model_name: v.vote for v in r.model_votes}
    rows.append(dict(
        true_label=y_s[i], true_cat=df_s['attack_cat'].iloc[i],
        verdict=r.final_verdict, cat=r.final_attack_cat,
        score=r.weighted_attack_score, conf=r.confidence,
        sev=r.severity, agree=r.agreement_score,
        RF=vm.get('RandomForest','?'), XGB=vm.get('XGBoost','?'),
        IF=vm.get('IsolationForest','?'), AE=vm.get('DenseAutoencoder','?'),
    ))

df = pd.DataFrame(rows)
df['correct'] = ((df.verdict=='ATTACK')&(df.true_label==1))|((df.verdict=='NORMAL')&(df.true_label==0))
print(f"Sample accuracy: {df.correct.mean():.4f}")


## 3. Voting Behaviour — Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for lbl, grp in df.groupby('true_label'):
    c = '#e74c3c' if lbl==1 else '#2ecc71'
    n = 'Attack' if lbl==1 else 'Normal'
    axes[0].hist(grp['score'], bins=50, alpha=0.55, color=c, density=True, label=n)
axes[0].axvline(ATTACK_THRESHOLD, color='black', lw=2, ls='--', label='Threshold=0.50')
axes[0].set(title='Score Distribution by True Label', xlabel='Weighted Attack Score', ylabel='Density')
axes[0].legend()

parts = axes[1].violinplot(
    [df[df.true_label==0].score.values, df[df.true_label==1].score.values],
    positions=[0,1], showmedians=True
)
for pc, c in zip(parts['bodies'], ['#2ecc71','#e74c3c']):
    pc.set_facecolor(c); pc.set_alpha(0.6)
axes[1].axhline(ATTACK_THRESHOLD, color='k', lw=1.5, ls='--', alpha=0.7)
axes[1].set_xticks([0,1]); axes[1].set_xticklabels(['Normal','Attack'])
axes[1].set(title='Score Separation', ylabel='Score')

plt.suptitle('Fig 1: Ensemble Voting Behaviour', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(PROJECT_ROOT/'reports'/'fig_voting.png', dpi=120, bbox_inches='tight')
plt.show()


## 4. Confidence Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for v, c in [('ATTACK','#e74c3c'),('NORMAL','#2ecc71')]:
    axes[0].hist(df[df.verdict==v]['conf'], bins=40, alpha=0.6, color=c, label=v, density=True)
axes[0].set(title='Confidence by Verdict', xlabel='Confidence', ylabel='Density')
axes[0].legend()

axes[1].hist(df[df.correct]['conf'], bins=35, alpha=0.6, color='#3498db', label='Correct', density=True)
axes[1].hist(df[~df.correct]['conf'], bins=35, alpha=0.6, color='#e74c3c', label='Incorrect', density=True)
axes[1].set(title='Confidence: Correct vs Incorrect', xlabel='Confidence', ylabel='Density')
axes[1].legend()

plt.suptitle('Fig 2: Confidence Distributions', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(PROJECT_ROOT/'reports'/'fig_confidence.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Mean conf correct: {df[df.correct].conf.mean():.4f} | incorrect: {df[~df.correct].conf.mean():.4f}")


## 5. Severity Distribution

In [ ]:
sev_order  = ['CRITICAL','HIGH','MEDIUM','LOW','N/A']
sev_colors = ['#c0392b','#e67e22','#f39c12','#3498db','#95a5a6']
atk = df[df.verdict=='ATTACK']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sev_c = atk['sev'].value_counts().reindex(sev_order, fill_value=0)
bars  = axes[0].bar(sev_c.index, sev_c.values, color=sev_colors, edgecolor='white')
for bar, n in zip(bars, sev_c.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, str(n), ha='center', fontsize=9)
axes[0].set(title='Severity Counts (ATTACK alerts)', xlabel='Severity', ylabel='Count')

palette = {'CRITICAL':'#c0392b','HIGH':'#e67e22','MEDIUM':'#f39c12','LOW':'#3498db'}
for s, c in palette.items():
    sub = atk[atk.sev==s]
    if len(sub)>0:
        axes[1].scatter(sub['conf'], sub['agree'], c=c, alpha=0.35, s=8, label=s, rasterized=True)
for thresh in [0.70, 0.85]:
    axes[1].axvline(thresh, color='grey', lw=1, ls='--', alpha=0.5)
    axes[1].axhline(thresh, color='grey', lw=1, ls='--', alpha=0.5)
axes[1].set(title='Confidence vs Agreement (by Severity)', xlabel='Confidence', ylabel='Agreement')
axes[1].legend(markerscale=3)

plt.suptitle('Fig 3: Severity Distribution', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(PROJECT_ROOT/'reports'/'fig_severity.png', dpi=120, bbox_inches='tight')
plt.show()


## 6. Agreement Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df['agree'], bins=30, color='#9b59b6', edgecolor='white', alpha=0.85)
axes[0].axvline(df['agree'].mean(), color='red', lw=2, ls='--', label=f"Mean={df.agree.mean():.3f}")
axes[0].set(title='Agreement Score — All Samples', xlabel='Agreement', ylabel='Count')
axes[0].legend()

for lbl, grp in df.groupby('true_label'):
    c = '#e74c3c' if lbl==1 else '#2ecc71'
    axes[1].hist(grp['agree'], bins=25, alpha=0.6, color=c,
                 label='Attack' if lbl==1 else 'Normal', density=True)
axes[1].set(title='Agreement by True Label', xlabel='Agreement', ylabel='Density')
axes[1].legend()

plt.suptitle('Fig 4: Agreement Analysis', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(PROJECT_ROOT/'reports'/'fig_agreement.png', dpi=120, bbox_inches='tight')
plt.show()

ag = metrics.get('agreement_stats', {})
for k, v in ag.items(): print(f"  {k}: {v}")


## 7. Ensemble vs Individual Model Comparison

In [ ]:
models = ['RandomForest','XGBoost','IsolationForest','DenseAutoencoder','Ensemble']
clrs   = {'RandomForest':'#3498db','XGBoost':'#9b59b6','IsolationForest':'#e67e22',
           'DenseAutoencoder':'#1abc9c','Ensemble':'#e74c3c'}

met_keys  = ['precision','recall','f1','false_positive_rate']
met_labels = ['Precision','Recall','F1 Score','FPR (↓ better)']

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, mk, ml in zip(axes.flatten(), met_keys, met_labels):
    vals  = [indiv.get(m,{}).get(mk, metrics.get(mk,0) if m=='Ensemble' else 0) for m in models]
    bars  = ax.bar(models, vals, color=[clrs[m] for m in models], edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.3f}', ha='center', fontsize=8.5)
    bars[-1].set_edgecolor('black'); bars[-1].set_linewidth(2)
    ax.set(title=ml, ylabel='Score', ylim=(0, min(1.0, max(vals)*1.3+0.05)))
    ax.tick_params(axis='x', rotation=18)

plt.suptitle('Fig 5: Ensemble vs Individual Models', fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT/'reports'/'fig_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


## 8. Vote Pattern Heatmap

In [ ]:
df['pattern'] = df.RF.str[0]+df.XGB.str[0]+df.IF.str[0]+df.AE.str[0]
pt = df.groupby(['pattern','verdict']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pt.div(pt.sum(axis=1),axis=0), annot=pt.values, fmt='d',
            cmap='RdYlGn', ax=ax, linewidths=0.5, cbar_kws={'label':'Fraction'})
ax.set(title='Vote Patterns → Verdicts\n(A=Attack N=Normal; RF·XGB·IF·AE)',
       xlabel='Final Verdict', ylabel='Vote Pattern')
plt.tight_layout()
plt.savefig(PROJECT_ROOT/'reports'/'fig_vote_patterns.png', dpi=120, bbox_inches='tight')
plt.show()

print("Top 10 patterns:")
for pat, cnt in df['pattern'].value_counts().head(10).items():
    print(f"  {pat}: {cnt:,} → {dict(df[df.pattern==pat].verdict.value_counts())}")


## 9. Summary & Viva Defence

### Performance (Full Test Set)
See Section 1 for metrics loaded from `reports/ensemble_evaluation.json`.

### Key Design Decisions

| Decision | Rationale |
|---|---|
| Weights 0.35/0.35/0.15/0.15 | Supervised models are label-validated; anomaly models complement with zero-day coverage |
| Threshold = 0.50 | Natural midpoint — no test-set calibration; preserves strict train/test separation |
| Severity uses confidence AND agreement | Prevents single high-weight model triggering CRITICAL; requires multi-model consensus |
| Category from higher raw_score model | Most confident supervised prediction wins tiebreak; fully auditable |
| No meta-learner | Arithmetic only — every alert score computable by hand on a whiteboard |

### SIEM Analogy
This ensemble mirrors a **correlation engine** (Splunk ES, QRadar offenses):
- **Data sources** → 4 ML models
- **Correlation rule** → weighted vote + threshold
- **Alert** → EnsembleResult with severity grading
- **Audit trail** → ModelVote records per alert


In [ ]:
# Final summary from the full-dataset report
print("=" * 55)
print(" ENSEMBLE FINAL SUMMARY (FULL TEST SET)")
print("=" * 55)
for k in ['accuracy','precision','recall','f1','false_positive_rate']:
    print(f"  {k:26s}: {metrics[k]:.4f}")
print(f"  TP={metrics['tp']:,}  TN={metrics['tn']:,}  FP={metrics['fp']:,}  FN={metrics['fn']:,}")
print()
ag = metrics.get('agreement_stats', {})
print(f"  Full agreement (all 4) : {ag.get('full_agreement_all4','N/A')}")
print(f"  Mean agreement         : {ag.get('mean_agreement','N/A')}")
sev = metrics.get('severity_distribution', {})
print()
print("  Severity distribution (ATTACK alerts):")
for s in ['CRITICAL','HIGH','MEDIUM','LOW']:
    print(f"    {s:10s}: {sev.get(s,0):,}")
